In [1]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# !aws configure  # skipped: interactive; not needed with local MLflow

In [3]:
# !aws configure  # skipped: interactive; not needed with local MLflow

In [4]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("file:./mlruns")

In [5]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location=('file:///C:/Users/HAI/Downloads/Youtube sentiment '
 'analysis/notebooks/mlruns/367774434750057793'), creation_time=1787583166164, experiment_id='367774434750057793', last_update_time=1787583166164, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}>

In [6]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

C:\Users\HAI\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import os
df = pd.read_csv(('tweets_preprocessing.csv' if os.path.exists('tweets_preprocessing.csv') else '/content/tweets_preprocessing.csv' if os.path.exists('/content/tweets_preprocessing.csv') else '../tweets_preprocessing.csv')).dropna()
df.shape

(54036, 14)

In [8]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

ngram_range = (1, 3)  # Trigram setting
max_features = 3000  # reduced for local memory/time  # Set max_features to 1000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

ros = RandomOverSampler(random_state=42)
X_train_vec, y_train = ros.fit_resample(X_train_vec, y_train)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

# Run the experiment for XGBoost
run_optuna_experiment()


[I 2026-08-24 20:53:28,331] A new study created in memory with name: no-name-ae1ffff8-1b73-4505-a63e-8bad77b35f7b


[I 2026-08-24 20:54:56,811] Trial 0 finished with value: 0.5830866025166543 and parameters: {'n_estimators': 70, 'learning_rate': 0.0006733169024607995, 'max_depth': 10}. Best is trial 0 with value: 0.5830866025166543.


[I 2026-08-24 20:55:11,557] Trial 1 finished with value: 0.5229459659511473 and parameters: {'n_estimators': 111, 'learning_rate': 0.0008131056872900969, 'max_depth': 3}. Best is trial 0 with value: 0.5830866025166543.


[I 2026-08-24 20:56:00,136] Trial 2 finished with value: 0.5622686898593634 and parameters: {'n_estimators': 89, 'learning_rate': 0.0015017966601511645, 'max_depth': 7}. Best is trial 0 with value: 0.5830866025166543.


[I 2026-08-24 20:57:47,231] Trial 3 finished with value: 0.5924315321983715 and parameters: {'n_estimators': 295, 'learning_rate': 0.004544877533824596, 'max_depth': 6}. Best is trial 3 with value: 0.5924315321983715.


...

[I 2026-08-24 21:22:14,885] Trial 26 finished with value: 0.7420429311621022 and parameters: {'n_estimators': 186, 'learning_rate': 0.05478968793719599, 'max_depth': 6}. Best is trial 10 with value: 0.7674870466321243.


[I 2026-08-24 21:23:49,391] Trial 27 finished with value: 0.8067172464840858 and parameters: {'n_estimators': 270, 'learning_rate': 0.0996245197288751, 'max_depth': 7}. Best is trial 27 with value: 0.8067172464840858.


[I 2026-08-24 21:25:36,182] Trial 28 finished with value: 0.786639526276832 and parameters: {'n_estimators': 292, 'learning_rate': 0.0584905675052072, 'max_depth': 7}. Best is trial 27 with value: 0.8067172464840858.


[I 2026-08-24 21:29:43,318] Trial 29 finished with value: 0.7513878608438194 and parameters: {'n_estimators': 298, 'learning_rate': 0.023615490123782987, 'max_depth': 10}. Best is trial 27 with value: 0.8067172464840858.


2026/08/24 21:31:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
